In [5]:
import json

from pathlib import Path
import numpy as np
import pandas as pd

In [6]:
# compute multimodal character continuity metrics (per-character profiles)

MODELS = ['human', 'claude45', 'gpt4o', 'internvl3', 'llama4scout', 'qwen3vl']
PROMPTS = ['original', 'large']
ROBERTA_EXCLUDED_STORY_IDS = [1214, 3761, 5047, 5540, 7195, 10499]

COREF_DIR = Path('/mimer/NOBACKUP/groups/naiss2025-22-1187/coherence-driven-humans/models/linkappend/data-out/conll_to_json')
CHARACTERS_FILE = Path('/mimer/NOBACKUP/groups/naiss2025-22-1187/coherence-driven-humans/data/sampled_60/sampled_60_stories.json')
CHARACTER_MAPPINGS_FILE = Path('/mimer/NOBACKUP/groups/naiss2025-22-1187/coherence-tacl/data/sampled_60/character_mappings.json')

In [7]:
def load_jsonlines(filepath):
    documents = []
    with open(filepath, 'r') as f:
        for line in f:
            documents.append(json.loads(line))
    return documents

def clean_human_original_data(doc):
    if len(doc['sentences']) > 0:
        last_sent = doc['sentences'][-1]
        if len(last_sent) >= 3 and last_sent[-3:] == ['[', 'SENT', ']']:
            doc['sentences'][-1] = last_sent[:-3]
            num_tokens_before_last = sum(len(sent) for sent in doc['sentences'][:-1])
            removed_token_start = num_tokens_before_last + len(last_sent) - 3
            new_clusters = []
            for cluster in doc['clusters']:
                new_mentions = [m for m in cluster if m[0] < removed_token_start]
                if new_mentions:
                    new_clusters.append(new_mentions)
            doc['clusters'] = new_clusters
    return doc

def extract_story_id(doc_key):
    if isinstance(doc_key, int):
        return doc_key
    s = str(doc_key).replace('story_', '')
    parts = s.split('_')
    if len(parts) >= 2 and parts[0] == 'doc':
        s = parts[1]
    try:
        return int(float(s))
    except (ValueError, TypeError):
        return None

def load_character_stories(filepath):
    story_characters = {}
    valid_story_ids = set()
    with open(filepath, 'r') as f:
        for line in f:
            entry = json.loads(line)
            story_id = entry['story_id']
            characters = entry.get('characters', [])
            char_names = set()
            for char in characters:
                if isinstance(char, dict) and 'name' in char:
                    name = char['name']
                    if name and name != '{}':
                        char_names.add(name.lower())
                elif isinstance(char, str) and char and char != '{}':
                    char_names.add(char.lower())
            story_characters[story_id] = char_names
            if len(char_names) > 0:
                valid_story_ids.add(story_id)
    return story_characters, valid_story_ids

def load_character_mappings(filepath):
    with open(filepath, 'r') as f:
        raw_mappings = json.load(f)
    mappings = {}
    for story_id_str, images in raw_mappings.items():
        story_id = int(story_id_str)
        image_chars = []
        for img_name, chars in images.items():
            idx = int(img_name.split('_img')[1].split('.')[0])
            names = {c['name'].lower() for c in chars if 'name' in c and c['name']}
            image_chars.append((idx, names))
        image_chars.sort(key=lambda x: x[0])
        mappings[story_id] = [chars for _, chars in image_chars]
    return mappings

In [8]:
def get_sentence_boundaries(doc):
    boundaries = []
    current = 0
    for sent in doc.get('sentences', []):
        boundaries.append((current, current + len(sent) - 1))
        current += len(sent)
    return boundaries

def get_characters_per_sentence(doc, valid_characters):
    sentences = doc.get('sentences', [])
    clusters = doc.get('clusters', [])
    boundaries = get_sentence_boundaries(doc)
    chars_per_sentence = [set() for _ in range(len(sentences))]
    all_tokens = [tok for sent in sentences for tok in sent]

    for cluster in clusters:
        cluster_texts = set()
        for mention in cluster:
            start, end = mention[0], mention[1]
            if start < len(all_tokens) and end < len(all_tokens):
                span_text = ' '.join(all_tokens[start:end+1]).lower()
                cluster_texts.add(span_text)
                for tok in all_tokens[start:end+1]:
                    cluster_texts.add(tok.lower())

        matched_char = None
        for char_name in valid_characters:
            if char_name in cluster_texts:
                matched_char = char_name
                break

        if matched_char:
            for mention in cluster:
                start_token = mention[0]
                for sent_idx, (sent_start, sent_end) in enumerate(boundaries):
                    if sent_start <= start_token <= sent_end:
                        chars_per_sentence[sent_idx].add(matched_char)
                        break

    return chars_per_sentence

def compute_span_continuity(presence_list):
    if not any(presence_list):
        return None
    first_idx = last_idx = None
    for i, present in enumerate(presence_list):
        if present:
            if first_idx is None:
                first_idx = i
            last_idx = i
    if first_idx is None:
        return None
    span_len = last_idx - first_idx + 1
    appearances = sum(1 for i in range(first_idx, last_idx + 1) if presence_list[i])
    return appearances / span_len

def compute_mcc_character_rows_for_story(doc, story_id, valid_characters, char_mappings):
    if story_id not in char_mappings or not valid_characters:
        return []
    chars_per_sentence = get_characters_per_sentence(doc, valid_characters)
    image_chars = char_mappings.get(story_id, [])
    if len(chars_per_sentence) == 0 or len(image_chars) == 0:
        return []

    rows = []
    for char in sorted(valid_characters):
        text_presence = [char in chars_per_sentence[i] for i in range(len(chars_per_sentence))]
        tc = compute_span_continuity(text_presence)

        image_presence = [char in image_chars[i] for i in range(len(image_chars))]
        ic = compute_span_continuity(image_presence)

        if tc is not None and ic is not None:
            mcc_raw_char = float(1 - abs(tc - ic))
            mcc_char = float(np.tanh(mcc_raw_char))
            missing_reason = ''
        elif tc is not None or ic is not None:
            mcc_raw_char = 0.0
            mcc_char = float(np.tanh(mcc_raw_char))
            missing_reason = '__ONE_SIDE_MISSING__'
        else:
            fill = 0.0
            mcc_raw_char = fill
            mcc_char = fill
            missing_reason = '__NO_SPAN_MATCH__'

        rows.append({
            'character_name': char,
            'tc': tc if tc is not None else np.nan,
            'ic': ic if ic is not None else np.nan,
            'mcc_raw_char': mcc_raw_char,
            'MCC_char': mcc_char,
            'missing_reason': missing_reason
        })

    return rows

In [9]:

def load_mcc_data():
    
    story_characters, valid_story_ids = load_character_stories(CHARACTERS_FILE)
    char_mappings = load_character_mappings(CHARACTER_MAPPINGS_FILE)
    rows = []

    for model in MODELS:
        for prompt in PROMPTS:
            seeds_to_load = ['seed42', 'seed43', 'seed44'] if (model == 'human' and prompt == 'large') else ['seed42']

            for seed in seeds_to_load:
                filepath = COREF_DIR / f"{model}_{prompt}_{seed}_test_snapshots__local_json-nopound_pred.jsonlines"
                if not filepath.exists():
                    continue
                docs = load_jsonlines(filepath)

                for doc in docs:
                    if model == 'human' and prompt == 'original':
                        doc = clean_human_original_data(doc)

                    story_id = extract_story_id(doc.get('doc_key'))
                    valid_chars = story_characters.get(story_id, set())

                    if story_id not in valid_story_ids or len(valid_chars) == 0:
                        fill = 0.0
                        rows.append({
                            'model': model, 'prompt': prompt, 'seed': seed, 'story_id': story_id,
                            'character_name': '__NO_ANNOTATION__',
                            'tc': np.nan, 'ic': np.nan,
                            'mcc_raw_char': fill, 'MCC_char': fill, 'missing_reason': '__NO_ANNOTATION__'
                        })
                        continue

                    char_rows = compute_mcc_character_rows_for_story(
                        doc, story_id, valid_chars, char_mappings
                    )

                    if len(char_rows) == 0:
                        fill = 0.0
                        rows.append({
                            'model': model, 'prompt': prompt, 'seed': seed, 'story_id': story_id,
                            'character_name': '__NO_MCC_MATCH__',
                            'tc': np.nan, 'ic': np.nan,
                            'mcc_raw_char': fill, 'MCC_char': fill, 'missing_reason': '__NO_MCC_MATCH__'
                        })
                        continue

                    for row in char_rows:
                        rows.append({
                            'model': model, 'prompt': prompt, 'seed': seed, 'story_id': story_id, **row
                        })

    return pd.DataFrame(rows)

def prepare_mcc_data(df, excluded_story_ids=None):
    # per-character output (human large averaged across seeds by story_id + character_name)
    out = []
    for model in MODELS:
        for prompt in PROMPTS:
            df_mp = df[(df['model'] == model) & (df['prompt'] == prompt)]
            if len(df_mp) == 0:
                continue
            if model == 'human' and prompt == 'large':
                df_avg = (
                    df_mp.groupby(['story_id', 'character_name'])[['tc', 'ic', 'mcc_raw_char', 'MCC_char']]
                    .mean()
                    .reset_index()
                )
                for _, row in df_avg.iterrows():
                    out.append({
                        'model': model, 'prompt': prompt,
                        'story_id': int(row['story_id']), 'character_name': row['character_name'],
                        'tc': row['tc'], 'ic': row['ic'],
                        'mcc_raw_char': row['mcc_raw_char'], 'MCC_char': row['MCC_char']
                    })
            else:
                df_seed42 = df_mp[df_mp['seed'] == 'seed42']
                for _, row in df_seed42.iterrows():
                    out.append({
                        'model': model, 'prompt': prompt,
                        'story_id': int(row['story_id']), 'character_name': row['character_name'],
                        'tc': row['tc'], 'ic': row['ic'],
                        'mcc_raw_char': row['mcc_raw_char'], 'MCC_char': row['MCC_char']
                    })
    result = pd.DataFrame(out)

    if excluded_story_ids:
        excluded_story_ids = {int(x) for x in excluded_story_ids}
        result = result[~result['story_id'].isin(excluded_story_ids)].copy()

    return result

In [10]:
df_mcc_raw = load_mcc_data()
df_mcc = prepare_mcc_data(df_mcc_raw)
df_mcc_54 = prepare_mcc_data(df_mcc_raw, excluded_story_ids=ROBERTA_EXCLUDED_STORY_IDS)

# origina, 60
story_original_60 = (
    df_mcc[df_mcc['prompt'] == 'original']
    .groupby(['model', 'story_id'])[['tc', 'ic', 'mcc_raw_char', 'MCC_char']]
    .mean()
    .reset_index()
    .rename(columns={'MCC_char': 'MCC'})
)

mcc_agg_original_60 = (
    story_original_60.groupby('model')
    .agg(
        tc_mean=('tc', 'mean'),
        ic_mean=('ic', 'mean'),
        mcc_raw_mean=('mcc_raw_char', 'mean'),
        MCC_mean=('MCC', 'mean'),
        MCC_std=('MCC', 'std'),
        count=('story_id', 'count')
    )
    .reset_index()
    .sort_values('model')
)

# original, 54
story_original_54 = (
    df_mcc_54[df_mcc_54['prompt'] == 'original']
    .groupby(['model', 'story_id'])[['tc', 'ic', 'mcc_raw_char', 'MCC_char']]
    .mean()
    .reset_index()
    .rename(columns={'MCC_char': 'MCC'})
)

mcc_agg_original_54 = (
    story_original_54.groupby('model')
    .agg(
        tc_mean=('tc', 'mean'),
        ic_mean=('ic', 'mean'),
        mcc_raw_mean=('mcc_raw_char', 'mean'),
        MCC_mean=('MCC', 'mean'),
        MCC_std=('MCC', 'std'),
        count=('story_id', 'count')
    )
    .reset_index()
    .sort_values('model')
)

# long, 54
story_large_54 = (
    df_mcc_54[df_mcc_54['prompt'] == 'large']
    .groupby(['model', 'story_id'])[['tc', 'ic', 'mcc_raw_char', 'MCC_char']]
    .mean()
    .reset_index()
    .rename(columns={'MCC_char': 'MCC'})
)

mcc_agg_large_54 = (
    story_large_54.groupby('model')
    .agg(
        tc_mean=('tc', 'mean'),
        ic_mean=('ic', 'mean'),
        mcc_raw_mean=('mcc_raw_char', 'mean'),
        MCC_mean=('MCC', 'mean'),
        MCC_std=('MCC', 'std'),
        count=('story_id', 'count')
    )
    .reset_index()
    .sort_values('model')
)

display(mcc_agg_original_60)

display(mcc_agg_original_54)

display(mcc_agg_large_54)

,model,tc_mean,ic_mean,mcc_raw_mean,MCC_mean,MCC_std,count
0,claude45,0.853575,0.828596,0.513315,0.409132,0.267862,60
1,gpt4o,0.871464,0.828596,0.549748,0.441300,0.281661,60
2,human,0.873045,0.828596,0.477309,0.381389,0.271206,60
3,internvl3,0.839790,0.828596,0.488585,0.401864,0.273968,60
4,llama4scout,0.930748,0.828596,0.116261,0.095177,0.213010,60
5,qwen3vl,0.818115,0.828596,0.556284,0.443075,0.280251,60


,model,tc_mean,ic_mean,mcc_raw_mean,MCC_mean,MCC_std,count
0,claude45,0.861466,0.840171,0.528220,0.420599,0.268718,54
1,gpt4o,0.867260,0.840171,0.572560,0.458836,0.279270,54
2,human,0.875699,0.840171,0.493306,0.392986,0.271860,54
3,internvl3,0.832697,0.840171,0.509539,0.417650,0.275661,54
4,llama4scout,0.924453,0.840171,0.119920,0.098198,0.218634,54
5,qwen3vl,0.819606,0.840171,0.587538,0.468133,0.279276,54


,model,tc_mean,ic_mean,mcc_raw_mean,MCC_mean,MCC_std,count
0,claude45,0.858506,0.840171,0.542518,0.430572,0.267028,54
1,gpt4o,0.859707,0.840171,0.629218,0.493148,0.293583,54
2,human,0.906973,0.840171,0.503065,0.403446,0.241348,54
3,internvl3,0.805642,0.840171,0.491825,0.403484,0.266717,54
4,llama4scout,0.889062,0.840171,0.290002,0.235590,0.279498,54
5,qwen3vl,0.825269,0.840171,0.568483,0.451172,0.280276,54


In [11]:
out_dir = Path('./analysis_data/mcc')
out_dir.mkdir(parents=True, exist_ok=True)

df_mcc_raw.to_csv(out_dir / 'mcc_metrics_raw.csv', index=False)
df_mcc.to_csv(out_dir / 'mcc_metrics.csv', index=False)
df_mcc_54.to_csv(out_dir / 'mcc_metrics_54.csv', index=False)

story_original_60.to_csv(out_dir / 'mcc_metrics_story_original_60.csv', index=False)
story_original_54.to_csv(out_dir / 'mcc_metrics_story_original_54.csv', index=False)
story_large_54.to_csv(out_dir / 'mcc_metrics_story_large_54.csv', index=False)
pd.concat([story_original_60, story_original_54, story_large_54], ignore_index=True).to_csv(out_dir / 'mcc_metrics_story.csv', index=False)

mcc_agg_original_60.to_csv(out_dir / 'mcc_metrics_agg_original_60.csv', index=False)
mcc_agg_original_54.to_csv(out_dir / 'mcc_metrics_agg_original_54.csv', index=False)
mcc_agg_large_54.to_csv(out_dir / 'mcc_metrics_agg_large_54.csv', index=False)
